# Oil Prices in 2026 — Act 5: What If the Model Could Read the News?

The [companion notebook](energy_oil_case_study.ipynb) showed that Prophet's rolling 30-day
forecast catastrophically missed the 2026 oil price surge — forecasting ~$61/bbl
while WTI hit $100. The model wasn't wrong in principle; it simply had no mechanism
for incorporating the geopolitical context that was publicly available at the time.

This notebook asks: **could a context-aware LLM forecaster have done better?**

We evaluate three key forecast origins in early 2026 using three methods side by side:
- **Prophet** (baseline — already computed, loaded from cache)
- **LLMP — no context** (Gemini 3 Flash, history only)
- **LLMP — with context** (same model + plausibly-knowable geopolitical context at each origin)

And we frame the comparison three ways:

| | Question type | Evaluation |
|---|---|---|
| **Act 5** | *Trajectory* — what will the 30-day price path look like? | MAE vs. actuals |
| **Act 6** | *Binary* — will price exceed a meaningful threshold in 30 days? | Calibration of P(exceed) |
| **Act 7** | *Causal* — what forces are the model anchoring on? | Qualitative reasoning audit |

In [1]:
from __future__ import annotations

import json
import logging
import os
import sys
import warnings
from pathlib import Path

import numpy as np
import pandas as pd
import plotly.graph_objects as go
import plotly.subplots as psp
from dotenv import load_dotenv

warnings.filterwarnings("ignore")
logging.getLogger("prophet").setLevel(logging.ERROR)

# ── Repo root: walk up from CWD until pyproject.toml is found ─────────────────
_cwd = Path(os.getcwd()).resolve()
REPO_ROOT = _cwd
while not (REPO_ROOT / "pyproject.toml").exists():
    if REPO_ROOT.parent == REPO_ROOT:
        REPO_ROOT = _cwd
        break
    REPO_ROOT = REPO_ROOT.parent

DATA_DIR = REPO_ROOT / "data"

for p in [str(REPO_ROOT / "implementations"), str(REPO_ROOT / "aieng-forecasting")]:
    if p not in sys.path:
        sys.path.insert(0, p)

load_dotenv(REPO_ROOT / ".env")

# ── Colour palette (matches companion notebook) ────────────────────────────────
CLR_HISTORY   = "#bdd7e7"
CLR_ACTUAL    = "#2171b5"   # solid blue — the truth
CLR_PROPHET   = "#636363"   # grey — the blind baseline
CLR_LLMP_BARE = "#fd8d3c"   # orange — LLM, history only
CLR_LLMP_CTX  = "#2ca02c"   # green — LLM, with context
CLR_CONFLICT  = "#d62728"   # red — conflict annotation
CONFLICT_DATE = pd.Timestamp("2026-03-01")

print(f"Repo root : {REPO_ROOT}")
print(f"Data dir  : {DATA_DIR}")
print("Setup complete.")

Repo root : /Users/ethanjackson/agentic-forecasting
Data dir  : /Users/ethanjackson/agentic-forecasting/data
Setup complete.


In [2]:
# ── WTI price history (same cache as companion notebook) ──────────────────────
PRICE_CACHE        = DATA_DIR / "wti_price_history.parquet"
PROPHET_CACHE      = DATA_DIR / "energy_case_study_forecasts_30d_daily_v3.parquet"
PROPHET_TRAJ_CACHE = DATA_DIR / "energy_prophet_trajectories.parquet"
LLMP_CACHE         = DATA_DIR / "energy_llmp_context_forecasts.parquet"

price_df = pd.read_parquet(PRICE_CACHE)
price_df.index = pd.DatetimeIndex(
    [pd.Timestamp(str(d)[:10]) for d in price_df.index]
)
price_df.index.name = "date"
price_df = price_df.sort_index()

prophet_df = pd.read_parquet(PROPHET_CACHE)
prophet_df["sim_day"]        = pd.to_datetime(prophet_df["sim_day"])
prophet_df["resolution_date"] = pd.to_datetime(prophet_df["resolution_date"])

print(f"WTI price history : {price_df.index[0].date()} → {price_df.index[-1].date()} ({len(price_df):,} days)")
print(f"Prophet forecasts : {prophet_df['sim_day'].min().date()} → {prophet_df['sim_day'].max().date()} ({len(prophet_df):,} rows)")

WTI price history : 2021-01-04 → 2026-05-01 (1,340 days)
Prophet forecasts : 2025-01-02 → 2026-04-01 (314 rows)


---

## The Setup

We pick **three forecast origins** in early 2026 — each representing a different
stage of the geopolitical escalation that drove WTI from ~$58 to $100+ between
January and April 2026.

| Origin | WTI at origin | Resolution date | Actual WTI at resolution | Prophet forecast | Context available |
|---|---|---|---|---|---|
| Jan 5, 2026 | $58 | Feb 4, 2026 | **$65** | $58 (inside CI) | Tensions building; OPEC+ cuts; insurance premiums rising |
| Feb 2, 2026 | $62 | Mar 4, 2026 | **$75** | $61 (miss — above CI) | Gulf of Oman incident; escalation fears; analyst upgrades |
| Mar 2, 2026 | $71 | Apr 1, 2026 | **$100** | $61 (catastrophic miss) | Conflict active; Strait of Hormuz blockade; IEA emergency session |

For each origin we ask: does an LLMP with access to publicly-available context
shift its forecast in the right direction — toward the actual outcome?

In [3]:
def compress_history(
    price_df: pd.DataFrame,
    as_of: pd.Timestamp,
    recent_window_months: int = 6,
) -> pd.DataFrame:
    """Return a token-efficient history: weekly averages for older data, daily for recent.

    Compresses ~1300 daily rows to ~200 rows while preserving the recent
    daily granularity that matters most for the LLM's short-horizon forecast.
    """
    hist = price_df[price_df.index <= as_of].copy()
    cutoff_daily = as_of - pd.DateOffset(months=recent_window_months)

    older = (
        hist[hist.index < cutoff_daily]
        .resample("W")
        .mean()
        .reset_index()
        .rename(columns={"date": "timestamp", "price": "value"})
    )
    recent = (
        hist[hist.index >= cutoff_daily]
        .reset_index()
        .rename(columns={"date": "timestamp", "price": "value"})
    )

    result = pd.concat([older, recent], ignore_index=True)
    result["timestamp"] = pd.to_datetime(result["timestamp"])
    return result[["timestamp", "value"]].sort_values("timestamp").reset_index(drop=True)


def prophet_row_at_origin(prophet_df: pd.DataFrame, origin: pd.Timestamp) -> pd.Series:
    """Return the Prophet forecast row whose sim_day is nearest to (on or after) origin."""
    candidates = prophet_df[prophet_df["sim_day"] >= origin]
    return candidates.iloc[0]


def resolution_price(price_df: pd.DataFrame, origin: pd.Timestamp, horizon_calendar_days: int = 30) -> tuple[pd.Timestamp, float]:
    """Return (resolution_date, actual_price) for a calendar-day horizon from origin."""
    target = origin + pd.Timedelta(days=horizon_calendar_days)
    row = price_df[price_df.index >= target].iloc[0]
    return row.name, float(row["price"])


# Sanity-check the three origins
ORIGINS = [
    pd.Timestamp("2026-01-05"),
    pd.Timestamp("2026-02-02"),
    pd.Timestamp("2026-03-02"),
]

# ── Shock experiment: 8 weekly origins spanning Feb–Mar 2026 ─────────────────
# 4 calm weeks followed by 4 shock weeks — a natural stress test.
SHOCK_ORIGINS = [
    pd.Timestamp("2026-02-02"),
    pd.Timestamp("2026-02-09"),
    pd.Timestamp("2026-02-17"),   # Tue (Mon = Presidents' Day holiday)
    pd.Timestamp("2026-02-23"),
    pd.Timestamp("2026-03-02"),
    pd.Timestamp("2026-03-09"),
    pd.Timestamp("2026-03-16"),
    pd.Timestamp("2026-03-23"),
]

print("Origin summary:")
for o in ORIGINS:
    price_at_origin = float(price_df[price_df.index >= o].iloc[0]["price"])
    res_date, res_price = resolution_price(price_df, o)
    p_row = prophet_row_at_origin(prophet_df, o)
    print(
        f"  {o.date()}  WTI=${price_at_origin:.2f}  "
        f"→ resolution {res_date.date()} actual=${res_price:.2f}  "
        f"prophet=${p_row['yhat']:.2f} [{p_row['yhat_lower']:.1f},{p_row['yhat_upper']:.1f}]  "
        f"inside_ci={p_row['inside_ci']}"
    )

Origin summary:
  2026-01-05  WTI=$58.32  → resolution 2026-02-04 actual=$65.14  prophet=$57.55 [49.5,65.5]  inside_ci=True
  2026-02-02  WTI=$62.14  → resolution 2026-03-04 actual=$74.66  prophet=$60.91 [52.8,69.4]  inside_ci=False
  2026-03-02  WTI=$71.23  → resolution 2026-04-01 actual=$100.12  prophet=$61.32 [53.3,69.3]  inside_ci=False


In [4]:
# ── Prophet full-trajectory forecasts for the three origins ──────────────────
#
# The existing Prophet cache stores only the terminal 30-calendar-day-ahead
# point per origin.  Here we re-run Prophet for just the three key origins and
# produce a 21-business-day trajectory fan — matching the LLMP output structure
# so the trajectory chart shows a like-for-like comparison between methods.
#
# Fitting 3 Prophet models takes ~10 s; results are cached to
# data/energy_prophet_trajectories.parquet so subsequent runs are instant.

from prophet import Prophet  # type: ignore[import-untyped]


def _fit_prophet_at_origin(price_df: pd.DataFrame, origin: pd.Timestamp) -> pd.DataFrame:
    """Fit one Prophet model on all data up to origin; return 21-business-day trajectory."""
    train_df = price_df.loc[:origin][["price"]].reset_index()
    train_df.columns = pd.Index(["ds", "y"])

    model = Prophet(
        interval_width=0.95,
        daily_seasonality=False,
        weekly_seasonality=False,
        yearly_seasonality=True,
        seasonality_mode="multiplicative",
    )
    model.fit(train_df)

    # Predict enough calendar days to cover 21 business days (≈ 31 calendar days)
    future = model.make_future_dataframe(periods=35, freq="D")
    pred   = model.predict(future).set_index("ds")

    bday_dates = pd.bdate_range(start=origin + pd.offsets.BDay(1), periods=21)
    rows = []
    for h, date in enumerate(bday_dates, start=1):
        cal_date = date.normalize()
        if cal_date in pred.index:
            row = pred.loc[cal_date]
        else:
            nearest_idx = int((pred.index - cal_date).abs().argmin())
            row = pred.iloc[nearest_idx]
        rows.append({
            "origin":        origin,
            "forecast_date": date,
            "horizon":       h,
            "yhat":          float(row["yhat"]),
            "yhat_lower":    float(row["yhat_lower"]),
            "yhat_upper":    float(row["yhat_upper"]),
        })

    return pd.DataFrame(rows)


def load_prophet_trajectories(
    price_df: pd.DataFrame,
    origins: list[pd.Timestamp],
    cache_path: Path,
) -> pd.DataFrame:
    """Load from cache or compute full Prophet trajectory for each origin."""
    if cache_path.exists():
        df = pd.read_parquet(cache_path)
        df["origin"]        = pd.to_datetime(df["origin"])
        df["forecast_date"] = pd.to_datetime(df["forecast_date"])
        print(f"Loaded {len(df)} Prophet trajectory rows from cache.")
        return df

    print("Fitting Prophet at 3 origins (~10 s)...")
    frames = []
    for origin in origins:
        print(f"  {origin.date()} ...", end=" ", flush=True)
        frames.append(_fit_prophet_at_origin(price_df, origin))
        print("done")

    df = pd.concat(frames, ignore_index=True)
    df.to_parquet(cache_path, index=False)
    print(f"Saved {len(df)} rows to {cache_path}")
    return df


prophet_traj_df = load_prophet_trajectories(price_df, ORIGINS, PROPHET_TRAJ_CACHE)
prophet_traj_df.head(6)

Loaded 63 Prophet trajectory rows from cache.

,origin,forecast_date,horizon,yhat,yhat_lower,yhat_upper
0,2026-01-05,2026-01-06,1,56.110597,47.564043,63.171764
1,2026-01-05,2026-01-07,2,56.226107,48.657983,64.782960
2,2026-01-05,2026-01-08,3,56.346479,48.503792,64.105882
3,2026-01-05,2026-01-09,4,56.470931,47.860042,64.401306
4,2026-01-05,2026-01-12,5,56.857824,48.616767,64.700857
5,2026-01-05,2026-01-13,6,56.986786,48.967157,65.217871


In [5]:
# ── Prophet trajectories for all 8 shock-experiment origins ──────────────────
# Reuse the same _fit_prophet_at_origin / load_prophet_trajectories helpers
# defined above, but with a separate cache so Act 5 data stays pristine.

PROPHET_SHOCK_TRAJ_CACHE = DATA_DIR / "energy_shock_prophet_trajectories.parquet"

prophet_shock_traj_df = load_prophet_trajectories(price_df, SHOCK_ORIGINS, PROPHET_SHOCK_TRAJ_CACHE)
print(
    f"Shock-origin Prophet trajectories: {len(prophet_shock_traj_df)} rows "
    f"across {prophet_shock_traj_df['origin'].nunique()} origins"
)
prophet_shock_traj_df.groupby("origin")["horizon"].agg(["min", "max", "count"])

Loaded 168 Prophet trajectory rows from cache.
Shock-origin Prophet trajectories: 168 rows across 8 origins


,min,max,count
origin,,,
2026-02-02,1,21,21
2026-02-09,1,21,21
2026-02-17,1,21,21
2026-02-23,1,21,21
2026-03-02,1,21,21
2026-03-09,1,21,21
2026-03-16,1,21,21
2026-03-23,1,21,21


In [6]:
# ── Context snippets — plausibly knowable on each origin date ─────────────────
# These represent the kind of information a professional energy analyst
# would have had access to from public sources: news, vessel-tracking
# services, futures data, and analyst reports published before the origin date.

ORIGIN_CONTEXTS: dict[str, dict] = {
    "2026-01-05": {
        "label": "Jan 5, 2026",
        "threshold_usd": 65.0,
        "context_text": (
            "As of January 5 2026:\n"
            "- WTI crude has been range-bound in the $56–62 band since October 2025 on soft "
            "demand signals and elevated US inventory builds.\n"
            "- OPEC+ is maintaining its current production-cut agreement through Q1 2026; "
            "no rollback has been signalled.\n"
            "- Iranian proxy forces conducted three separate attacks on US logistics assets "
            "in Iraq and Syria in Q4 2025. US-Iran tensions are elevated but have not "
            "escalated to direct military exchange.\n"
            "- Lloyd's of London hull-war insurance premiums for tankers transiting the "
            "Gulf of Oman have risen approximately 15% since September 2025.\n"
            "- The WTI NYMEX forward curve is in mild backwardation: front month $58, "
            "6-month forward approximately $56.\n"
            "- EIA weekly report (Dec 31 2025): US crude inventories 8% below the 5-year "
            "seasonal average."
        ),
    },
    "2026-02-02": {
        "label": "Feb 2, 2026",
        "threshold_usd": 72.0,
        "context_text": (
            "As of February 2 2026:\n"
            "- WTI gained approximately 7% in January, closing near $62, driven by "
            "escalating Persian Gulf tensions.\n"
            "- A US Navy escort mission in the Gulf of Oman was intercepted by Iranian "
            "fast-attack boats on January 28. No shots fired, but the incident was "
            "widely reported and prompted a diplomatic protest from Washington.\n"
            "- OPEC+ called an emergency ministerial consultation for February 10 amid "
            "concerns about supply-chain disruption risk; no production change announced yet.\n"
            "- Goldman Sachs revised its 2026 WTI price target upward to $70–85 in a "
            "February 1 research note, citing a 'geopolitical risk premium re-rating'.\n"
            "- Vessel-tracking data shows tanker transits through the Strait of Hormuz "
            "down approximately 15% week-over-week, as operators seek alternative routings.\n"
            "- Brent/WTI spread widened to $4.50, the largest since early 2024, as "
            "European buyers began bidding up non-Gulf grades.\n"
            "- US intelligence officials stated publicly that Iranian military assets "
            "have been repositioned closer to the Strait of Hormuz."
        ),
    },
    "2026-03-02": {
        "label": "Mar 2, 2026",
        "threshold_usd": 85.0,
        "context_text": (
            "As of March 2 2026:\n"
            "- The US conducted direct airstrikes on Iranian oil-infrastructure targets "
            "on March 1 2026 in response to an Iranian attack on a US carrier group "
            "in the Gulf of Oman on February 26.\n"
            "- Iran declared a partial blockade of the Strait of Hormuz effective "
            "March 1; approximately 20% of global seaborne oil supply transits the Strait.\n"
            "- WTI surged from $62 on February 2 to $71 by March 2 — a 14% move in "
            "one month — and front-month futures gapped up a further $4 at Monday open.\n"
            "- The IEA called an emergency ministerial meeting for March 5 to consider "
            "releasing strategic petroleum reserves.\n"
            "- Saudi Aramco issued force majeure declarations on several customer contracts; "
            "Saudi Arabia activated its emergency supply protocols.\n"
            "- Goldman Sachs issued an updated note on March 1 with a new 2026 WTI target "
            "of $95–115 and flagging a tail-risk scenario of $130 if the blockade persists "
            "beyond 60 days.\n"
            "- WTI NYMEX forward curve has swung into sharp backwardation: front month "
            "$71, 6-month forward $62, signalling market expectation of eventual resolution."
        ),
    },
}

print("Context snippets defined for:", list(ORIGIN_CONTEXTS.keys()))

Context snippets defined for: ['2026-01-05', '2026-02-02', '2026-03-02']


In [7]:
# ── Run LLMP forecasts (or load from cache) ───────────────────────────────────
#
# We use the LLMP module's internals directly so we can supply a pre-compressed
# history DataFrame rather than going through a DataService.  This is appropriate
# for a playground notebook; a production version would use a registered adapter.
#
# Each origin × 2 variants (bare / with-context) = 6 LLM calls.
# Results are cached to data/energy_llmp_context_forecasts.parquet.

from aieng.forecasting.methods.llm_processes.continuous import (
    ContinuousLLMPredictorConfig,
    _build_system_prompt,
    _build_user_prompt,
    _quantiles_per_step,
    _sample_trajectories,
    _stack_trajectories,
)
from aieng.forecasting.methods.llm_processes.base import serialize_history
from aieng.forecasting.evaluation.prediction import STANDARD_QUANTILES
from aieng.forecasting.evaluation.task import ForecastingTask
from aieng.forecasting.data.models import SeriesMetadata


MODEL       = "gemini/gemini-3-flash-preview"
N_SAMPLES   = 20
HORIZON_B   = 21   # ~21 business days ≈ 30 calendar days
PRECISION   = 2

_WTI_TASK = ForecastingTask(
    task_id="wti_crude_30d",
    target_series_id="wti_crude",
    horizons=list(range(1, HORIZON_B + 1)),
    frequency="B",
    description=(
        "WTI crude oil front-month futures price (USD/bbl), "
        "30 trading-day ahead probabilistic forecast. "
        "Forecast the daily closing price for each of the next "
        f"{HORIZON_B} business days."
    ),
)

_WTI_META = SeriesMetadata(
    series_id="wti_crude",
    description="WTI crude oil front-month futures (CL=F, Adj Close)",
    source="Yahoo Finance",
    units="USD/bbl",
    frequency="B",
)


def _run_one_forecast(
    price_df: pd.DataFrame,
    origin: pd.Timestamp,
    context_text: str | None,
    context_tag: str,
) -> dict:
    """Run LLMP at a single origin and return a dict of arrays."""
    history_df = compress_history(price_df, origin)
    history_str = serialize_history(history_df, precision=PRECISION)

    forecast_start = origin + pd.offsets.BDay(1)
    forecast_end   = origin + pd.offsets.BDay(HORIZON_B)

    system_prompt = _build_system_prompt()
    user_prompt   = _build_user_prompt(
        _WTI_TASK, history_str, _WTI_META,
        forecast_start, forecast_end, HORIZON_B,
        context_text=context_text,
    )

    cfg = ContinuousLLMPredictorConfig(
        model=MODEL,
        n_samples=N_SAMPLES,
        temperature=1.0,
        reasoning_effort="disable",
        context_text=context_text,
        context_tag=context_tag,
    )

    parsed, cost_usd, in_tok, out_tok, failures = _sample_trajectories(
        cfg=cfg, system_prompt=system_prompt, user_prompt=user_prompt
    )
    samples = _stack_trajectories([t.values for t in parsed], n_steps=HORIZON_B)
    q_grid  = _quantiles_per_step(samples)   # (HORIZON_B, len(STANDARD_QUANTILES))

    dates = pd.bdate_range(start=origin + pd.offsets.BDay(1), periods=HORIZON_B)

    rows = []
    for h_idx in range(HORIZON_B):
        row: dict = {
            "origin":      origin,
            "context_tag": context_tag,
            "forecast_date": dates[h_idx],
            "horizon":     h_idx + 1,
            "median":      float(q_grid[h_idx, STANDARD_QUANTILES.index(0.50)]),
            "cost_usd":    cost_usd,
        }
        for qi, q in enumerate(STANDARD_QUANTILES):
            row[f"q{int(q * 100):02d}"] = float(q_grid[h_idx, qi])
        rows.append(row)

    print(
        f"    origin={origin.date()} tag={context_tag:12s} "
        f"cost=${cost_usd:.4f}  failures={failures}/{N_SAMPLES}"
    )
    return {"rows": rows, "samples": samples.tolist()}


def run_all_forecasts(price_df: pd.DataFrame, cache_path: Path) -> pd.DataFrame:
    """Run (or load) all 6 LLMP forecasts; return a single flat DataFrame."""
    if cache_path.exists():
        df = pd.read_parquet(cache_path)
        df["origin"]        = pd.to_datetime(df["origin"])
        df["forecast_date"] = pd.to_datetime(df["forecast_date"])
        print(f"Loaded {len(df)} LLMP forecast rows from cache.")
        return df

    all_rows: list[dict] = []
    print("Running LLMP forecasts (6 API calls)...")
    for origin in ORIGINS:
        key = origin.strftime("%Y-%m-%d")
        ctx = ORIGIN_CONTEXTS[key]
        for tag, text in [("bare", None), ("context", ctx["context_text"])]:
            result = _run_one_forecast(price_df, origin, text, tag)
            all_rows.extend(result["rows"])

    df = pd.DataFrame(all_rows)
    df.to_parquet(cache_path, index=False)
    print(f"Saved {len(df)} rows to {cache_path}")
    return df


llmp_df = run_all_forecasts(price_df, LLMP_CACHE)
print(f"\nOrigins: {sorted(llmp_df['origin'].dt.date.unique())}")
print(f"Tags:    {sorted(llmp_df['context_tag'].unique())}")
llmp_df.head(6)

Loaded 126 LLMP forecast rows from cache.

Origins: [datetime.date(2026, 1, 5), datetime.date(2026, 2, 2), datetime.date(2026, 3, 2)]
Tags:    ['bare', 'context']


,origin,context_tag,forecast_date,horizon,median,cost_usd,q05,q10,q20,q30,q40,q50,q60,q70,q80,q90,q95
0,2026-01-05,bare,2026-01-06,1,58.450,0.07594,58.1500,58.420,58.450,58.450,58.450,58.450,58.450,58.450,58.624,58.650,58.6595
1,2026-01-05,bare,2026-01-07,2,58.820,0.07594,58.3955,58.410,58.620,58.720,58.780,58.820,58.820,58.820,59.120,59.123,59.1530
2,2026-01-05,bare,2026-01-08,3,59.080,0.07594,58.1935,58.300,58.590,58.769,58.846,59.080,59.150,59.150,59.150,59.180,59.4515
3,2026-01-05,bare,2026-01-09,4,58.835,0.07594,58.0910,58.120,58.394,58.700,58.724,58.835,58.874,59.158,59.226,59.384,59.4315
4,2026-01-05,bare,2026-01-12,5,59.165,0.07594,57.8850,57.937,58.426,58.550,59.052,59.165,59.232,59.323,59.420,59.787,59.8515
5,2026-01-05,bare,2026-01-13,6,59.280,0.07594,57.9150,58.400,58.686,58.959,59.092,59.280,59.368,59.680,59.904,60.120,60.1200


---

## Act 5 — Trajectory: Can the Model See the Move Coming?

Each panel below shows the 30-day forecast fan from one origin date.
Three forecast traces:
- **Grey** — Prophet (statistical baseline; history only)
- **Orange** — LLMP, history only (same information as Prophet, different model family)
- **Green** — LLMP with context (public geopolitical information available on that date)

The **solid blue line** is the realized WTI price (actual outcome).
Shading shows 50% and 90% credible intervals for the LLMP forecasts.

In [8]:
def _add_fan(
    fig: go.Figure,
    dates: "pd.Series",
    lower: "pd.Series",
    upper: "pd.Series",
    median: "pd.Series",
    color: str,
    opacity_band: float,
    name: str,
    show_legend: bool,
    legendgroup: str,
    row: int,
    col: int,
) -> None:
    """Add a forecast fan (CI band + median line) to a subplot panel."""
    # CI shading (fill between lower and upper)
    fig.add_trace(
        go.Scatter(
            x=pd.concat([dates, dates[::-1]]),
            y=pd.concat([lower, upper[::-1]]),
            fill="toself",
            fillcolor=color,
            opacity=opacity_band,
            line=dict(width=0),
            mode="lines",
            showlegend=False,
        ),
        row=row, col=col,
    )
    # Median line
    fig.add_trace(
        go.Scatter(
            x=dates, y=median,
            line=dict(color=color, width=2),
            name=name if show_legend else None,
            showlegend=show_legend,
            legendgroup=legendgroup,
        ),
        row=row, col=col,
    )


def make_trajectory_figure(
    price_df: pd.DataFrame,
    prophet_traj_df: pd.DataFrame,
    llmp_df: pd.DataFrame,
    origins: list[pd.Timestamp],
) -> go.Figure:
    """Three-column subplot: one panel per origin, like-for-like trajectory fan comparison.

    All three methods (Prophet, LLMP-bare, LLMP-with-context) are shown as a
    CI-band + median line over the full 21-business-day forecast horizon, making
    the comparison visually consistent.
    """
    labels = [ORIGIN_CONTEXTS[o.strftime("%Y-%m-%d")]["label"] for o in origins]

    fig = psp.make_subplots(
        rows=1, cols=3,
        subplot_titles=labels,
        shared_yaxes=False,
        horizontal_spacing=0.06,
    )

    for col, origin in enumerate(origins, start=1):
        # ── History (60 days pre-origin) ──────────────────────────────────────
        hist_start = origin - pd.Timedelta(days=60)
        hist = price_df[price_df.index >= hist_start].loc[:origin]
        fig.add_trace(
            go.Scatter(
                x=hist.index, y=hist["price"],
                line=dict(color=CLR_HISTORY, width=2),
                name="History" if col == 1 else None,
                showlegend=(col == 1),
                legendgroup="history",
            ),
            row=1, col=col,
        )

        # ── Actuals post-origin ────────────────────────────────────────────────
        res_date, _ = resolution_price(price_df, origin)
        actuals = price_df[(price_df.index > origin) & (price_df.index <= res_date)]
        fig.add_trace(
            go.Scatter(
                x=actuals.index, y=actuals["price"],
                line=dict(color=CLR_ACTUAL, width=2.5),
                name="Actual" if col == 1 else None,
                showlegend=(col == 1),
                legendgroup="actual",
            ),
            row=1, col=col,
        )

        # ── Prophet trajectory fan ─────────────────────────────────────────────
        pt = prophet_traj_df[prophet_traj_df["origin"] == origin].sort_values("forecast_date")
        if not pt.empty:
            _add_fan(
                fig,
                dates=pt["forecast_date"],
                lower=pt["yhat_lower"],
                upper=pt["yhat_upper"],
                median=pt["yhat"],
                color=CLR_PROPHET,
                opacity_band=0.15,
                name="Prophet 95% CI",
                show_legend=(col == 1),
                legendgroup="prophet",
                row=1, col=col,
            )

        # ── LLMP fans (bare + context) ─────────────────────────────────────────
        for tag, clr, name in [
            ("bare",    CLR_LLMP_BARE, "LLMP — history only"),
            ("context", CLR_LLMP_CTX,  "LLMP — with context"),
        ]:
            sub = llmp_df[(llmp_df["origin"] == origin) & (llmp_df["context_tag"] == tag)].sort_values("forecast_date")
            if sub.empty:
                continue

            # 90% CI (outer, light)
            _add_fan(
                fig,
                dates=sub["forecast_date"],
                lower=sub["q05"],
                upper=sub["q95"],
                median=sub["median"] if "median" in sub.columns else sub["q50"],
                color=clr,
                opacity_band=0.10,
                name=name,
                show_legend=(col == 1),
                legendgroup=f"llmp_{tag}",
                row=1, col=col,
            )
            # 50% CI (inner, darker)
            if "q25" in sub.columns and "q75" in sub.columns:
                fig.add_trace(
                    go.Scatter(
                        x=pd.concat([sub["forecast_date"], sub["forecast_date"][::-1]]),
                        y=pd.concat([sub["q25"], sub["q75"][::-1]]),
                        fill="toself",
                        fillcolor=clr,
                        opacity=0.20,
                        line=dict(width=0),
                        mode="lines",
                        showlegend=False,
                    ),
                    row=1, col=col,
                )

        # ── Origin marker ──────────────────────────────────────────────────────
        fig.add_vline(
            x=origin.timestamp() * 1000,
            line=dict(color="#888", dash="dash", width=1),
            row=1, col=col,
        )

    fig.update_layout(
        title=dict(
            text="30-Day WTI Forecast Trajectories — Prophet vs. LLMP vs. LLMP + Context",
            font=dict(size=15),
        ),
        height=420,
        width=1200,
        legend=dict(orientation="h", y=-0.18),
        template="plotly_white",
        margin=dict(t=60, b=90),
    )
    fig.update_yaxes(title_text="WTI (USD/bbl)", col=1)
    return fig


make_trajectory_figure(price_df, prophet_traj_df, llmp_df, ORIGINS).show()

The January origin stays within Prophet's CI. February and March do not — those are the
cases where context makes the difference. Prophet's 30-day median ends up ~$61 for all three
regardless of what's happening in the world. The LLMP with context shifts upward when the
news warrants it.

---

## Act 6 — Binary Forecasting: Will WTI Rise by More Than $5 This Week?

Act 5 showed that trajectory forecasts diverge sharply once context enters the picture.
Here we pin it to a single, evaluable binary question:

> **P(WTI day-5 close > today's price + $5/bbl)** — 8 weekly origins, Feb 2 – Mar 23, 2026

**Why Prophet fails here structurally:** Prophet always forecasts reversion toward its
long-run trend (~$63). When WTI is at $71–$95, Prophet's implied probability that prices
rise *further* by $5+ is essentially zero. But they did — by $23 on Mar 2 and $15 on Mar 23.
The Analyst Agent reads the news and disagrees.

| Origin | WTI | 5-day Δ | Outcome |
|---|---|---|---|
| Feb 2 | $62.14 | +$2.22 | No |
| Feb 9 | $64.36 | −$2.03 | No |
| Feb 17 | $62.33 | +$3.30 | No |
| Feb 23 | $66.31 | +$4.92 | No |
| Mar 2 | $71.23 | **+$23.54** | **Yes — upward shock** |
| Mar 9 | $94.77 | −$1.27 | No |
| Mar 16 | $93.50 | −$5.37 | No |
| Mar 23 | $88.13 | **+$14.75** | **Yes — upward shock** |

In [9]:
import asyncio
import json
import re

import scipy.interpolate
import scipy.stats
from google.adk.agents import LlmAgent
from google.adk.runners import Runner
from google.adk.sessions import InMemorySessionService
from google.adk.tools import google_search
from google.genai import types as genai_types
from google.genai.types import GenerateContentConfig
import litellm

# ── Parameters ────────────────────────────────────────────────────────────────
SHOCK_THRESHOLD      = 5.0   # day-5 close > today + $5 = "upward shock"
SHOCK_HORIZON        = 5     # business days (1 trading week)
SHOCK_ANALYST_CACHE  = DATA_DIR / "energy_upshock_analyst_forecasts.json"
SHOCK_CONTEXT_CACHE  = DATA_DIR / "energy_upshock_news_context.json"

# ── Ground truth ──────────────────────────────────────────────────────────────
def check_shock_outcome(
    price_df: pd.DataFrame,
    origin: pd.Timestamp,
    threshold: float,
    horizon_bdays: int,
) -> tuple[int, float]:
    """Return (outcome, delta): outcome=1 if day-H close > origin_price + threshold."""
    origin_price = float(price_df[price_df.index >= origin].iloc[0]["price"])
    future_days  = price_df[price_df.index > origin].iloc[:horizon_bdays]
    delta        = float(future_days.iloc[-1]["price"]) - origin_price
    return (1 if delta > threshold else 0), delta

# ── Prophet P(upward shock) ───────────────────────────────────────────────────
def prophet_prob_shock(
    prophet_traj_sub: pd.DataFrame,
    origin_price: float,
    threshold: float,
    horizon: int = SHOCK_HORIZON,
) -> float:
    """P(price_h > origin + threshold) from Prophet's 95% CI (Gaussian approx)."""
    row = prophet_traj_sub[prophet_traj_sub["horizon"] == horizon]
    if row.empty:
        return float("nan")
    row = row.iloc[0]
    sigma = (float(row["yhat_upper"]) - float(row["yhat_lower"])) / (2 * 1.96)
    if sigma <= 0:
        return 1.0 if float(row["yhat"]) > origin_price + threshold else 0.0
    return float(np.clip(
        1.0 - scipy.stats.norm.cdf(origin_price + threshold, loc=float(row["yhat"]), scale=sigma),
        0.0, 1.0,
    ))

# ── Context Agent (Google ADK + google_search) ────────────────────────────────
_CONTEXT_SYSTEM = """\
You are an oil market intelligence specialist with access to web search.

CRITICAL TEMPORAL CONSTRAINT — you are simulating the perspective of an analyst
as of {cutoff_date}.
- Include ONLY information that was publicly available BEFORE {cutoff_date}.
- EXCLUDE any events, market moves, or data from {cutoff_date} or later.
- If a search result appears to post-date the cutoff, skip it entirely.
- This constraint is absolute. Violating it would corrupt the forecast.

Search for and summarise oil-market-relevant information focused on:
- WTI/Brent crude price level and recent trend
- OPEC+ production decisions and supply outlook
- Geopolitical risks in the Persian Gulf, Middle East, key shipping lanes
- US Strategic Petroleum Reserve and energy policy signals
- Notable tanker/shipping incidents or supply chain disruption signals
- Any published analyst forecasts or unusual price-target revisions

Focus especially on factors that could cause a *sudden large move* in WTI crude
over the next 5–10 days. Return a concise structured markdown summary (3–5 paragraphs).\
"""

async def _retrieve_oil_context_async(cutoff_date: str) -> str:
    """Run the Context Agent and return its oil-market summary as of cutoff_date."""
    _APP = "oil-context-agent"
    session_svc = InMemorySessionService()
    agent = LlmAgent(
        name="oil_context_agent",
        instruction=_CONTEXT_SYSTEM.format(cutoff_date=cutoff_date),
        tools=[google_search],
        model="gemini-3-flash-preview",
        generate_content_config=GenerateContentConfig(temperature=0.1, max_output_tokens=2048),
    )
    runner = Runner(agent=agent, app_name=_APP, session_service=session_svc)
    session = await session_svc.create_session(app_name=_APP, user_id="nb")
    prompt = (
        f"Provide an oil market intelligence briefing as of {cutoff_date}. "
        f"Focus on supply risks, OPEC+ policy, Persian Gulf geopolitics, and any factors "
        f"that could cause a sudden large price move in WTI crude in the next 5–10 days. "
        f"IMPORTANT: only use information available before {cutoff_date}."
    )
    content = genai_types.Content(role="user", parts=[genai_types.Part(text=prompt)])
    async for event in runner.run_async(user_id="nb", session_id=session.id, new_message=content):
        if event.is_final_response() and event.content and event.content.parts:
            return event.content.parts[0].text or ""
    return ""

# ── Analyst Agent (litellm / Gemini Flash) ────────────────────────────────────
_ANALYST_SYSTEM = """\
You are an expert oil market analyst making short-term probabilistic forecasts.

You will receive:
  1. Recent WTI crude oil price history (daily, most recent last)
  2. An oil market intelligence briefing with a strict temporal cutoff

Your task: estimate P(up) — the probability that WTI will close MORE THAN
${threshold}/bbl HIGHER than today's price at the end of {horizon} trading days.

This is a directional upside question only.

Be calibrated:
- No unusual upside catalyst → base rate ~10-15%.
- Escalating but unconfirmed geopolitical risk → 20-40%.
- Confirmed major supply disruption actively driving prices higher → 60-85%.

Respond ONLY in valid JSON with exactly these fields:
{{
  "probability_up": <float between 0 and 1>,
  "direction_bias": "<up|down|neutral>",
  "reasoning": "<2-4 sentences>",
  "key_signals": ["<signal 1>", "<signal 2>", "<signal 3>"],
  "confidence": "<high|medium|low>"
}}
Output ONLY the JSON object. No other text.\
"""

def _run_analyst(
    history_str: str,
    news_context: str,
    origin_date: str,
    origin_price: float,
) -> dict:
    """Analyst Agent: reason from price history + news context → structured dict."""
    system = _ANALYST_SYSTEM.format(horizon=SHOCK_HORIZON, threshold=int(SHOCK_THRESHOLD))
    target = origin_price + SHOCK_THRESHOLD
    user_prompt = (
        f"### WTI Price History (ending {origin_date})\n\n"
        f"{history_str}\n\n"
        f"---\n"
        f"### Oil Market Briefing (as of {origin_date})\n\n"
        f"{news_context}\n\n"
        f"---\n"
        f"Current WTI price: ${origin_price:.2f}/bbl on {origin_date}.\n"
        f"Estimate P(WTI closes above ${target:.2f} in {SHOCK_HORIZON} trading days)."
    )
    response = litellm.completion(
        model="gemini/gemini-3-flash-preview",
        messages=[{"role": "system", "content": system}, {"role": "user", "content": user_prompt}],
        temperature=0.2,
        response_format={"type": "json_object"},
    )
    raw = response.choices[0].message.content or "{}"
    raw = re.sub(r"^```(?:json)?\s*", "", raw.strip())
    raw = re.sub(r"\s*```$", "", raw)
    try:
        return json.loads(raw)
    except json.JSONDecodeError:
        m = re.search(r'"probability_up"\s*:\s*([\d.]+)', raw)
        prob = float(m.group(1)) if m else float("nan")
        return {"reasoning": raw[:500], "probability_up": prob,
                "key_signals": [], "direction_bias": "?", "confidence": "?"}

print("Agent definitions loaded.")
print(f"  SHOCK_THRESHOLD : +${SHOCK_THRESHOLD:.0f}/bbl  |  SHOCK_HORIZON : {SHOCK_HORIZON} bdays")
print(f"  Question        : P(WTI day-5 close > today + ${SHOCK_THRESHOLD:.0f})")
print(f"  Context Agent   : Google ADK + google_search  (gemini-3-flash-preview, temporal cutoff)")
print(f"  Analyst Agent   : gemini-3-flash-preview  →  JSON  (probability_up + reasoning)")

Agent definitions loaded.
  SHOCK_THRESHOLD : +$5/bbl  |  SHOCK_HORIZON : 5 bdays
  Question        : P(WTI day-5 close > today + $5)
  Context Agent   : Google ADK + google_search  (gemini-3-flash-preview, temporal cutoff)
  Analyst Agent   : gemini-3-flash-preview  →  JSON  (probability_up + reasoning)


In [10]:
# ── Run pipeline (or load from cache) ────────────────────────────────────────
# For each of 8 weekly shock-experiment origins:
#   (1) Context Agent fetches news; (2) Analyst Agent estimates P(shock).
# Top-level `await` works in Jupyter — uses the kernel's running event loop.

if SHOCK_ANALYST_CACHE.exists() and SHOCK_CONTEXT_CACHE.exists():
    with open(SHOCK_ANALYST_CACHE) as f:
        shock_analyst_results: list[dict] = json.load(f)
    with open(SHOCK_CONTEXT_CACHE) as f:
        shock_news_contexts: dict[str, str] = json.load(f)
    print(f"Loaded {len(shock_analyst_results)} cached shock-experiment forecasts.")
else:
    shock_news_contexts = {}
    shock_analyst_results = []

    for origin in SHOCK_ORIGINS:
        key          = origin.strftime("%Y-%m-%d")
        cutoff       = (origin - pd.Timedelta(days=1)).strftime("%Y-%m-%d")
        origin_price = float(price_df[price_df.index >= origin].iloc[0]["price"])

        print(f"\n{'─' * 60}")
        print(f"Origin {key}  (context cutoff: {cutoff})  WTI ${origin_price:.2f}")

        print("  [1/2] Context Agent — searching for oil market intelligence ...")
        ctx = await _retrieve_oil_context_async(cutoff)
        shock_news_contexts[key] = ctx
        print(f"        {len(ctx):,} chars retrieved")

        print(f"  [2/2] Analyst Agent — reasoning about P(shock > ${SHOCK_THRESHOLD:.0f}) ...")
        hist     = compress_history(price_df, origin)
        hist_str = serialize_history(hist, precision=2)
        result   = _run_analyst(hist_str, ctx, key, origin_price)
        result["origin"] = key
        shock_analyst_results.append(result)
        p_up = result.get("probability_up", float("nan"))
        p_str = f"{p_up:.2f}" if isinstance(p_up, float) else str(p_up)
        print(f"        P(up>+${SHOCK_THRESHOLD:.0f})={p_str}  confidence={result.get('confidence', '?')}")

    with open(SHOCK_ANALYST_CACHE, "w") as f:
        json.dump(shock_analyst_results, f, indent=2)
    with open(SHOCK_CONTEXT_CACHE, "w") as f:
        json.dump(shock_news_contexts, f, indent=2)
    print("\nSaved to cache.")

# ── Assemble binary_df: Prophet + Analyst Agent + Always-50% baseline ─────────
shock_analyst_by_origin = {r["origin"]: r for r in shock_analyst_results}

binary_rows = []
for origin in SHOCK_ORIGINS:
    key          = origin.strftime("%Y-%m-%d")
    origin_price = float(price_df[price_df.index >= origin].iloc[0]["price"])
    outcome, max_move = check_shock_outcome(price_df, origin, SHOCK_THRESHOLD, SHOCK_HORIZON)

    # Prophet: P(shock) from Gaussian approximation to 5-day trajectory CI
    pt_sub = prophet_shock_traj_df[prophet_shock_traj_df["origin"] == origin]
    p_prob = prophet_prob_shock(pt_sub, origin_price, SHOCK_THRESHOLD)

    # Analyst Agent
    a      = shock_analyst_by_origin.get(key, {})
    a_prob = float(a.get("probability_up", float("nan")))

    for method, prob in [("Prophet", p_prob), ("Analyst Agent", a_prob), ("Always 50%", 0.5)]:
        binary_rows.append({
            "origin": key,
            "origin_price": origin_price,
            "max_move": max_move,
            "outcome": outcome,
            "method": method,
            "prob": prob,
            "brier": (prob - outcome) ** 2,
            "reasoning": a.get("reasoning") if method == "Analyst Agent" else None,
            "key_signals": a.get("key_signals", []) if method == "Analyst Agent" else [],
            "confidence": a.get("confidence") if method == "Analyst Agent" else None,
            "direction_bias": a.get("direction_bias") if method == "Analyst Agent" else None,
        })

binary_df = pd.DataFrame(binary_rows)
binary_df["origin_dt"] = pd.to_datetime(binary_df["origin"])
binary_df = binary_df.sort_values(["origin_dt", "method"]).reset_index(drop=True)

# ── Quick sanity check ─────────────────────────────────────────────────────────
print("\n── Binary forecast summary ─────────────────────────────────────────────")
summary = binary_df.pivot_table(
    index="origin", columns="method", values=["prob", "brier"], aggfunc="first"
)
print(summary.to_string())

Loaded 8 cached shock-experiment forecasts.

── Binary forecast summary ─────────────────────────────────────────────
                brier                                   prob                            
method     Always 50% Analyst Agent       Prophet Always 50% Analyst Agent       Prophet
origin                                                                                  
2026-02-02       0.25        0.0144  6.582261e-04        0.5          0.12  2.565592e-02
2026-02-09       0.25        0.0484  3.039773e-04        0.5          0.22  1.743494e-02
2026-02-17       0.25        0.0324  8.373381e-03        0.5          0.18  9.150618e-02
2026-02-23       0.25        0.1444  1.198964e-03        0.5          0.38  3.462606e-02
2026-03-02       0.25        0.0324  9.964893e-01        0.5          0.82  1.756918e-03
2026-03-09       0.25        0.4624  1.492722e-27        0.5          0.68  3.863576e-14
2026-03-16       0.25        0.1764  7.371441e-21        0.5          0.42  8.585

In [11]:
# ── Display: one clean forecast card per origin ────────────────────────────────
from IPython.display import display, Markdown as MD

def _verdict(a_prob: float, outcome: int, delta: float, threshold: float) -> str:
    """Plain-English verdict: was the agent right, and by how much?"""
    if outcome == 1:
        if a_prob >= 0.50:
            return f"✅ **Correct** — predicted shock, shock happened (+${delta:.2f})"
        else:
            return f"❌ **Missed** — predicted no shock, but price rose +${delta:.2f}"
    else:
        if a_prob < 0.50:
            return f"✅ **Correct** — predicted no shock, price moved only {delta:+.2f}"
        else:
            return f"❌ **False alarm** — predicted shock, but price only moved {delta:+.2f}"

def _conf_bar(confidence: str) -> str:
    bars = {"high": "████░", "medium": "███░░", "low": "██░░░"}
    return bars.get(confidence.lower(), "?????") if confidence else "?????"

for origin in SHOCK_ORIGINS:
    key          = origin.strftime("%Y-%m-%d")
    label        = origin.strftime("%b %-d, %Y")
    a            = shock_analyst_by_origin.get(key, {})
    ctx          = shock_news_contexts.get(key, "*(no context retrieved)*")
    origin_price = float(price_df[price_df.index >= origin].iloc[0]["price"])
    outcome, _   = check_shock_outcome(price_df, origin, SHOCK_THRESHOLD, SHOCK_HORIZON)
    delta        = float(price_df[price_df.index > origin].iloc[:SHOCK_HORIZON].iloc[-1]["price"]) - origin_price

    a_prob      = float(a.get("probability_up", float("nan")))
    confidence  = a.get("confidence", "?")
    reasoning   = a.get("reasoning", "N/A")
    signals     = a.get("key_signals", [])
    brier       = (a_prob - outcome) ** 2

    outcome_badge = "🔴 **SHOCK**" if outcome else "🟢 **No shock**"
    prob_bar = int(round(a_prob * 10))
    prob_vis = "█" * prob_bar + "░" * (10 - prob_bar)

    cutoff = (origin - pd.Timedelta(days=1)).strftime("%Y-%m-%d")

    md = f"""
---
### {label} — WTI ${origin_price:.2f} /bbl

| | |
|---|---|
| **Prediction** | P(up > +${SHOCK_THRESHOLD:.0f}) = **{a_prob:.0%}**  `{prob_vis}` |
| **Confidence** | {confidence.title() if confidence else "?"}  `{_conf_bar(confidence)}` |
| **Rationale** | {reasoning} |
| **Key signals** | {" · ".join(signals) if signals else "—"} |
| **Actual outcome** | {outcome_badge} — price moved **{delta:+.2f} /bbl** over 5 days |
| **Verdict** | {_verdict(a_prob, outcome, delta, SHOCK_THRESHOLD)} |
| **Brier score** | {brier:.3f} {"🟢 good" if brier < 0.10 else "🟡 ok" if brier < 0.25 else "🔴 poor"} *(0 = perfect, 0.25 = random)* |

<details><summary>📰 Oil market context used (cutoff: {cutoff}) — click to expand</summary>

{ctx}

</details>
"""
    display(MD(md))


---
### Feb 2, 2026 — WTI $62.14 /bbl

| | |
|---|---|
| **Prediction** | P(up > +$5) = **12%**  `█░░░░░░░░░` |
| **Confidence** | Medium  `███░░` |
| **Rationale** | The market is currently weighed down by a significant projected global supply surplus of 2.1 to 4.0 million bpd, which creates a strong fundamental headwind. While the briefing identifies upside risks such as SPR refill announcements and geopolitical tensions, the recent sharp decline from $65 to $62.14 suggests bearish momentum is currently dominant. A $5 rally in five trading days would require an immediate and major catalyst to overcome the prevailing supply glut narrative. |
| **Key signals** | Projected 2026 global oil surplus · U.S. SPR buy-back policy rhetoric · Red Sea maritime security levels |
| **Actual outcome** | 🟢 **No shock** — price moved **+2.22 /bbl** over 5 days |
| **Verdict** | ✅ **Correct** — predicted no shock, price moved only +2.22 |
| **Brier score** | 0.014 🟢 good *(0 = perfect, 0.25 = random)* |

<details><summary>📰 Oil market context used (cutoff: 2026-02-01) — click to expand</summary>

### **Oil Market Intelligence Briefing: February 1, 2026**

**Price Levels and Market Sentiment**
As of February 1, 2026, WTI crude is trading in a bearish corridor between **$53 and $57 per barrel**, while Brent remains pressured near the **$60 mark**. The market is currently dominated by a "supply glut" narrative, with the EIA and IEA both forecasting a significant global surplus of **2.1 to 4.0 million barrels per day (bpd)** for the first half of 2026. This bearishness is driven by record-breaking non-OPEC+ production growth from the U.S., Brazil, and Guyana, which has consistently outpaced sluggish demand growth in the petrochemical and aviation sectors.

**OPEC+ Production Strategy**
In response to deteriorating market conditions, OPEC+ has implemented a **"strategic pause"** for the first quarter of 2026. The alliance, led by Saudi Arabia and Russia, recently confirmed it will freeze all planned production hikes through March to prevent a total price collapse. While this move was intended to provide a floor, internal tensions are rising; the UAE’s recent signals regarding its long-term production capacity and the ongoing "compensation" disputes with over-producers like Iraq and Kazakhstan have created a fragile unity. Analysts warn that if prices dip below $50 WTI, the group may be forced into an emergency meeting to consider deeper, involuntary cuts.

**Geopolitical Risks and Shipping Lanes**
The Persian Gulf and Red Sea remain in a state of **"fragile equilibrium."** While large-scale disruptions have been avoided in the opening weeks of 2026, maritime security remains at a "Critical" level. Houthi activity in the Red Sea continues to force roughly 50% of Suez-bound traffic to reroute around the Cape of Good Hope, maintaining a persistent "risk premium" in freight and insurance costs. Furthermore, the U.S. administration has recently ratcheted up rhetoric against Iran and Venezuela, with new warnings regarding airspace and maritime "contraband" checks. Any tactical miscalculation in the Strait of Hormuz—which still carries 20% of global supply—remains the primary "black swan" candidate for a sudden $10–$15 price spike.

**U.S. Policy and Strategic Petroleum Reserve (SPR)**
The U.S. Strategic Petroleum Reserve currently sits at approximately **411 million barrels**, its lowest level in decades. The Trump administration has declared refilling the SPR a "Department-level priority" for 2026, seeking to capitalize on the current price dip. With WTI trading well below the administration's unofficial **$75–$80 target range**, market participants are closely watching for a large-scale "buy-back" announcement. Such a move would provide a significant psychological floor and could trigger a sharp short-term rally as the government competes with commercial refiners for physical barrels.

**5–10 Day Volatility Outlook**
The risk of a **sudden large move** in the next 5–10 days is skewed to the upside. While the fundamental outlook is bearish, the market is "short-heavy," making it vulnerable to a short squeeze. Key triggers to watch include:
1.  **Persian Gulf Escalation:** Any confirmed strike on energy infrastructure or a "closure" threat in the Strait of Hormuz.
2.  **SPR Intervention:** A formal DOE solicitation to purchase 10M+ barrels for the reserve.
3.  **OPEC+ Rhetoric:** Surprise signals from Riyadh regarding an early end to the "strategic pause" in favor of deeper cuts.
Conversely, a break below the **$50 support level** for WTI could trigger automated sell-offs if the 2026 surplus projections are further revised upward in upcoming February agency reports.

</details>



---
### Feb 9, 2026 — WTI $64.36 /bbl

| | |
|---|---|
| **Prediction** | P(up > +$5) = **22%**  `██░░░░░░░░` |
| **Confidence** | Medium  `███░░` |
| **Rationale** | The market is currently dominated by a bearish supply glut narrative, which has led to heavy short positioning that is vulnerable to a squeeze. While the 'Trump Floor' at $60/bbl provides support, a $5 move in five days likely requires a kinetic geopolitical catalyst in the Persian Gulf, where tensions are currently simmering but unconfirmed. Given the 15% rise in maritime insurance premiums and recent drone sightings, the tail risk of a sudden spike is elevated above base rates. |
| **Key signals** | Persian Gulf insurance premium trends · SPR solicitation volumes · OPEC+ internal cohesion rumors |
| **Actual outcome** | 🟢 **No shock** — price moved **-2.03 /bbl** over 5 days |
| **Verdict** | ✅ **Correct** — predicted no shock, price moved only -2.03 |
| **Brier score** | 0.048 🟢 good *(0 = perfect, 0.25 = random)* |

<details><summary>📰 Oil market context used (cutoff: 2026-02-08) — click to expand</summary>

### **Oil Market Intelligence Briefing**
**Date:** February 8, 2026
**Subject:** WTI Crude Outlook and Supply Risk Assessment

#### **Current Price Levels and Market Sentiment**
As of early February 2026, **WTI crude is trading in the $62–$65/bbl range**, with Brent hovering near $68/bbl. The prevailing market sentiment is characterized by a "bearish overhang" following the International Energy Agency’s (IEA) January 21 report, which warned of a looming **global supply glut** that could peak at 4.5 million b/d by Q2 2026. This surplus is driven primarily by surging production from the "Americas Quintet" (U.S., Canada, Brazil, Guyana, and Argentina). However, prices have found a firm floor due to the Trump administration’s aggressive mandate to refill the **U.S. Strategic Petroleum Reserve (SPR)**, which currently stands at approximately 415 million barrels. The Department of Energy (DOE) is actively purchasing 1–2 million barrels per month, signaling a "Trump Floor" near $60/bbl.

#### **OPEC+ Policy and Supply Outlook**
OPEC+ is currently in a **production pause for Q1 2026**, having reaffirmed in late 2025 that they would suspend planned output increases for January, February, and March to counter seasonal demand weakness. While the group is officially keeping 3.6 million b/d off the market through year-end, internal cohesion is under scrutiny. Rumors regarding a potential **UAE exit from OPEC** have intensified, following the UAE’s successful bid for a higher production baseline. Market participants are also closely monitoring "compensation cuts" from Iraq and Kazakhstan, who have historically struggled with compliance. Any signal that OPEC+ might abandon its Q1 pause early to reclaim market share from U.S. shale would likely trigger a sharp downward move in WTI toward the $55 level.

#### **Geopolitical Risks and Shipping Disruptions**
While the market is currently focused on oversupply, **geopolitical "tail risks" in the Persian Gulf** are at their highest levels in years. Tensions between the U.S./Israel and Iran are described by analysts as "simmering at a crossroads." Although the Strait of Hormuz remains open as of today, insurance premiums for VLCCs (Very Large Crude Carriers) transiting the Gulf have risen by 15% over the last fortnight due to a series of "shadow war" incidents and drone sightings near regional energy infrastructure. The market is currently pricing in a very low probability of a full blockade; however, with 21 million b/d (20% of global supply) passing through this chokepoint, the disconnect between "glut pricing" and "conflict risk" is extreme.

#### **5–10 Day Outlook: Triggers for a Sudden Move**
The potential for a **sudden $5–$10 spike in WTI** over the next 10 days is high, primarily due to the market's heavy short positioning on the "supply glut" narrative. Key triggers to watch include:
*   **Maritime Incidents:** Any kinetic strike or seizure of a tanker in the Strait of Hormuz would cause an immediate short squeeze, as the IEA’s projected surplus cannot compensate for a disruption of that magnitude.
*   **U.S. Policy Shifts:** Unexpected signals regarding new tariffs on energy-exporting partners could dampen demand expectations further, while a sudden acceleration in SPR solicitations would tighten the physical prompt market.
*   **OPEC+ Rhetoric:** Ahead of the March 2026 window, any "leaks" regarding the extension of the production pause into Q2 would be viewed as a bullish signal, potentially lifting WTI back toward $70/bbl.

</details>



---
### Feb 17, 2026 — WTI $62.33 /bbl

| | |
|---|---|
| **Prediction** | P(up > +$5) = **18%**  `██░░░░░░░░` |
| **Confidence** | Medium  `███░░` |
| **Rationale** | A $5 increase in five trading days represents an approximate 8% rally, which is historically rare without a confirmed supply shock or major geopolitical escalation. While tightened sanctions on Russia and Persian Gulf tensions provide upside risk, the market is currently dominated by bearish IEA projections of a 4.5 million b/d surplus and seasonal demand weakness. The 'Trump Floor' created by SPR buying at $60-$63 provides strong support but does not inherently act as a catalyst for a sharp upward spike. |
| **Key signals** | IEA projected Q2 supply glut · OPEC+ March meeting rumors · Russian sanction escalation headlines |
| **Actual outcome** | 🟢 **No shock** — price moved **+3.30 /bbl** over 5 days |
| **Verdict** | ✅ **Correct** — predicted no shock, price moved only +3.30 |
| **Brier score** | 0.032 🟢 good *(0 = perfect, 0.25 = random)* |

<details><summary>📰 Oil market context used (cutoff: 2026-02-16) — click to expand</summary>

As of February 16, 2026, the crude oil market is characterized by a fragile stability, with WTI trading in the **$62–$65 range**. While prices have recovered from the five-month lows of $60 seen in late October 2025, the market remains under pressure from a looming "supply glut" projected for the second quarter of 2026. Sentiment is currently balanced between OPEC+ production discipline and a surge in output from the "Americas quintet" (US, Canada, Brazil, Guyana, and Argentina), which the IEA recently warned could lead to a massive 4.5 million b/d surplus by mid-year.

**OPEC+ Policy and Supply Outlook**
OPEC+ has taken a "proactively defensive" stance to start the year. Following their January 4, 2026, virtual meeting, the group reaffirmed a **pause in production increases** for February and March. This decision was driven by seasonal demand weakness and the need to offset the 2.9 million b/d added to the market in the latter half of 2025. Key members, including Iraq and the UAE, are currently under a strict "compensation schedule" to rectify previous overproduction, a move intended to signal group unity despite internal pressure from the UAE to increase its baseline capacity.

**Geopolitical Risks and Shipping**
Geopolitical risk premiums are currently centered on the friction surrounding **tightened sanctions on Russian oil majors** (Rosneft and Lukoil) and ongoing "wobbles" in the Persian Gulf. While the Strait of Hormuz remains open and functional, market participants are closely monitoring the "trade talks" in Washington, which have introduced volatility regarding the future of energy-related tariffs and sanctions. Any sudden breakdown in these diplomatic channels or a retaliatory move by Russia against the latest US/UK sanctions could trigger a rapid $5–$10 spike in WTI as shorts cover.

**US Energy Policy and the SPR**
Under the current administration, the **Strategic Petroleum Reserve (SPR)** has shifted from a source of supply to a significant source of demand. As of mid-February 2026, the SPR stands at approximately **415 million barrels**, up from 395 million a year ago. The Department of Energy (DOE) has made refilling the reserve a "national security priority," with a standing strategy to purchase crude whenever WTI dips toward the $60–$63 level. This "Trump Floor" is providing a critical psychological support level for WTI, preventing a deeper slide despite bearish IEA demand forecasts.

**5–10 Day Volatility Outlook**
Over the next 5–10 days, WTI is at risk of a **sudden large move** triggered by two primary factors:
1.  **Downside Risk:** If upcoming weekly inventory data confirms the IEA’s "bloated inventory" thesis or if Chinese demand indicators for Q1 underperform, WTI could test the $60 support level.
2.  **Upside Risk:** The market is highly sensitive to "headline risk" regarding the Persian Gulf. With several OPEC+ members meeting again on March 1, any rumors of an extended pause or deeper voluntary cuts to combat the Q2 glut could spark a sharp relief rally toward $70. Analysts at Goldman Sachs and JPMorgan have warned that while fundamentals are "structurally weak," the low level of OECD commercial inventories makes the market prone to extreme volatility on any physical supply disruption.

</details>



---
### Feb 23, 2026 — WTI $66.31 /bbl

| | |
|---|---|
| **Prediction** | P(up > +$5) = **38%**  `████░░░░░░` |
| **Confidence** | Medium  `███░░` |
| **Rationale** | The market is currently pricing in a significant geopolitical risk premium due to imminent threats of military escalation in the Persian Gulf and potential disruptions to the Strait of Hormuz. While physical fundamentals suggest a surplus, the intelligence briefing indicates a high probability of a supply shock within the 5-10 day window. A $5 move is well within the range of volatility expected if kinetic action or major shipping incidents occur. |
| **Key signals** | Military posturing in the Persian Gulf · Strait of Hormuz shipping notices · OPEC+ production cut extensions |
| **Actual outcome** | 🟢 **No shock** — price moved **+4.92 /bbl** over 5 days |
| **Verdict** | ✅ **Correct** — predicted no shock, price moved only +4.92 |
| **Brier score** | 0.144 🟡 ok *(0 = perfect, 0.25 = random)* |

<details><summary>📰 Oil market context used (cutoff: 2026-02-22) — click to expand</summary>

### **Oil Market Intelligence Briefing**
**Date:** February 22, 2026  
**Subject:** Supply Risks and Geopolitical Volatility Outlook (5–10 Day Window)

#### **Price Trend and Market Sentiment**
As of late February 2026, WTI crude is trading in the **$68–$71 range**, showing a distinct bullish tilt after a period of relative stability. While the IEA’s February 12 report highlighted a supply-demand surplus in early Q1 due to record non-OPEC+ production in 2025, prices have recently decoupled from these fundamentals. A "geopolitical risk premium" of approximately $8–$10 has begun to embed into Brent and WTI benchmarks over the last 10 days. This shift is driven by intensifying rumors of a shift in U.S. and Israeli posture toward Iran, moving the market from a "bearish surplus" narrative to one of "imminent supply shock" anticipation.

#### **OPEC+ Policy and Supply Outlook**
The OPEC+ alliance remains the primary floor for prices, having solidified its strategy in late 2025. On December 5, 2025, the group formally **extended its current production cuts through December 31, 2026**. This includes the 1.65 million bpd voluntary adjustments originally announced in 2023. While the group had initially planned to phase out some cuts in early 2026, the January 2026 ministerial monitoring committee (JMMC) signaled a "cautious and flexible" approach, effectively pausing any production increases. This discipline has successfully offset the 1.2 million bpd supply plunge seen in January caused by severe winter weather disruptions in North America and outages in Kazakhstan.

#### **Geopolitical Risks and the Persian Gulf**
The most critical factor for the next 5–10 days is the **deteriorating security situation in the Persian Gulf**. Intelligence signals and diplomatic rhetoric from Washington and Jerusalem suggest that the "shadow war" with Tehran is reaching a breaking point. Market participants are pricing in a high probability of a direct military escalation before the end of the month. The primary concern is a de facto closure or significant disruption of the **Strait of Hormuz**, which handles roughly 20% of global oil consumption. Any kinetic action against Iranian infrastructure or a retaliatory blockade by the IRGC would immediately strand 11–16 million bpd of Gulf exports, a disruption for which there is no global equivalent in spare capacity.

#### **US Strategic Petroleum Reserve (SPR) and Policy**
The U.S. energy policy under the current administration has shifted aggressively toward **refilling the Strategic Petroleum Reserve**. As of the week ending February 13, 2026, the SPR stands at **415.4 million barrels**, up approximately 15% from 2024 levels. While the Department of Energy (DOE) continues to award small-scale refill contracts (most recently for 1 million barrels at the Bryan Mound site), the administration has signaled that the SPR is now being held as a "strategic weapon" rather than a price-control tool. This suggests that the U.S. may be less willing to release emergency barrels to dampen minor price spikes, preferring to keep the reserve as a buffer against the looming Middle Eastern conflict.

#### **5–10 Day Forecast: Sudden Move Triggers**
The potential for a **sudden $20–$30 move in WTI** over the next 5–10 days is at its highest level in years. The "trigger" is expected to be a military strike or a major shipping incident in the Strait of Hormuz. If hostilities commence, analysts project WTI could breach **$100/bbl** almost instantly, as global inventories—despite recent builds—cannot compensate for a total Persian Gulf shut-in. Conversely, if a diplomatic "de-escalation" occurs, WTI would likely collapse back toward its fundamental support at **$60/bbl**. Traders should watch for any "Notice to Mariners" in the Gulf or sudden shifts in U.S. carrier strike group positions as the primary lead indicators for this move.

</details>



---
### Mar 2, 2026 — WTI $71.23 /bbl

| | |
|---|---|
| **Prediction** | P(up > +$5) = **82%**  `████████░░` |
| **Confidence** | High  `████░` |
| **Rationale** | The assassination of Iran's Supreme Leader and the subsequent blockade of the Strait of Hormuz represent a catastrophic geopolitical supply shock. With 20% of global oil supply effectively halted and major shipping lines suspending transit, the market is pricing in a massive war premium that historically leads to double-digit price spikes. A $5 move from the current $71.23 level is a conservative threshold given that analysts are already targeting the $80-$85 range in the immediate term. |
| **Key signals** | Closure of the Strait of Hormuz · US-Israeli strikes on Iranian nuclear/military infrastructure · Suspension of Persian Gulf transit by major shipping conglomerates |
| **Actual outcome** | 🔴 **SHOCK** — price moved **+23.54 /bbl** over 5 days |
| **Verdict** | ✅ **Correct** — predicted shock, shock happened (+$23.54) |
| **Brier score** | 0.032 🟢 good *(0 = perfect, 0.25 = random)* |

<details><summary>📰 Oil market context used (cutoff: 2026-03-01) — click to expand</summary>

### **Oil Market Intelligence Briefing: March 1, 2026**

**Immediate Catalyst: The "Black Saturday" Strikes and Geopolitical Escalation**
The primary driver for the coming week is the massive geopolitical shock following the February 28, 2026, coordinated air strikes by U.S. and Israeli forces against Iranian military and nuclear infrastructure. As of today, Sunday, March 1, global markets are bracing for a violent "gap up" at the Monday open. WTI, which closed Friday at approximately $66–$67/bbl, is expected to surge toward the $80–$85 range immediately. The reported assassination of Iran’s Supreme Leader during the strikes has moved the conflict from a regional proxy war to an existential state-level confrontation, placing a massive "war premium" back into crude prices that had previously been trading on bearish oversupply fundamentals.

**Geopolitical Flashpoint: Strait of Hormuz and Shipping Blockades**
The most critical risk to supply is the effective closure of the Strait of Hormuz. Following the strikes, the Iranian Revolutionary Guard Corps (IRGC) has issued warnings forbidding passage through the waterway, which handles roughly 20% of global seaborne oil (approx. 20 mb/d). Major shipping conglomerates, including Maersk and MSC, have already announced the suspension of all transits as of this morning. Reports of sea mines and the boarding of merchant vessels in the Gulf of Oman suggest a total maritime blockade is imminent. Unlike the Red Sea disruptions of 2024, there is no viable alternative for the volume of crude and LNG exiting the Persian Gulf; even a partial 10-day closure could remove 100 million barrels from the global market, potentially pushing WTI toward triple digits within the next 5–10 days.

**OPEC+ Policy: The March 1 Virtual Meeting**
In a move to signal market stability, OPEC+ members (led by Saudi Arabia and Russia) met virtually today, March 1. The group decided to proceed with a scheduled production adjustment of 206,000 barrels per day for April 2026, continuing the gradual unwinding of voluntary cuts. However, the group reaffirmed "full flexibility" to pause or reverse these hikes if market conditions deteriorate. While the production increase is technically bearish, it is being viewed by analysts as a "placeholder" decision; the real focus is on whether Saudi Arabia will activate its East-West Pipeline to bypass the Strait of Hormuz, though its 7 mb/d capacity cannot fully offset a total Gulf blockade.

**US SPR and Energy Policy Signals**
The Trump administration enters this crisis with the Strategic Petroleum Reserve (SPR) at approximately 415.4 million barrels (roughly 58% capacity), following a year of steady refilling efforts. While the administration has signaled a commitment to "energy dominance," the market is hyper-focused on whether an emergency SPR release will be coordinated with the IEA to blunt the expected price spike. Analysts at Goldman Sachs and Morgan Stanley have already revised their March targets, with some projecting Brent could exceed $110/bbl if the Hormuz closure persists. Any signal from the White House regarding a massive "emergency exchange" (loan) of crude could be the only factor capable of preventing WTI from testing its 2022 highs of $120+ in the coming week.

**5–10 Day Risk Assessment: Sudden Move Factors**
*   **Upside (Bullish):** Iranian retaliatory strikes on Saudi Aramco’s "Abqaiq" facility or UAE processing plants; confirmed mining of the Strait of Hormuz; or a formal declaration of *force majeure* by Kuwaiti or Iraqi exporters.
*   **Downside (Bearish):** An immediate U.S.-led "Freedom of Navigation" operation successfully escorting tankers through the Strait; or a surprise announcement of a 100M+ barrel global SPR release.
*   **Trend:** The technical "descending channel" that dominated 2025 has been shattered. Expect extreme volatility with a heavy bullish bias as the market prices in the largest potential supply disruption in history.

</details>



---
### Mar 9, 2026 — WTI $94.77 /bbl

| | |
|---|---|
| **Prediction** | P(up > +$5) = **68%**  `███████░░░` |
| **Confidence** | Medium  `███░░` |
| **Rationale** | The closure of the Strait of Hormuz has removed 14 mb/d of supply, creating a massive structural deficit that outweighs the minor OPEC+ production increases. WTI has demonstrated extreme momentum, rising over $23 in the last five trading days, making a further $5 move highly probable under current volatility. While a rumored 400-million-barrel IEA release poses a significant downside risk, the immediate 'war premium' and physical shortages are likely to sustain the upward trajectory in the short term. |
| **Key signals** | Strait of Hormuz operational status · IEA emergency release announcement · Integrity of Saudi East-West Pipeline |
| **Actual outcome** | 🟢 **No shock** — price moved **-1.27 /bbl** over 5 days |
| **Verdict** | ❌ **False alarm** — predicted shock, but price only moved -1.27 |
| **Brier score** | 0.462 🔴 poor *(0 = perfect, 0.25 = random)* |

<details><summary>📰 Oil market context used (cutoff: 2026-03-08) — click to expand</summary>

### **Oil Market Intelligence Briefing: March 8, 2026**

**Price Action and Market Sentiment**
As of March 8, 2026, WTI crude is experiencing unprecedented vertical volatility, trading in a wide range between **$125 and $145 per barrel**. This follows a catastrophic price spike initiated in February 2026 after the escalation of the "Iran War" involving U.S. and Israeli forces. The market has transitioned from a period of relative stability in early January (where WTI averaged $65) to a state of extreme "war premium" pricing. Sentiment is dominated by fear of a prolonged global energy deficit, as the effective closure of the Strait of Hormuz has removed nearly **14 million barrels per day (mb/d)** of Middle Eastern supply from the global balance.

**OPEC+ Policy and Supply Outlook**
On March 1, 2026, the eight key OPEC+ members (led by Saudi Arabia and Russia) announced they would increase production by **206,000 bpd starting in April 2026**. While this move was intended to signal a commitment to market stability, traders have largely dismissed it as a "drop in the bucket" given the scale of the current disruption. The group is attempting to resume the unwinding of voluntary cuts first announced in 2023, but the physical inability to export oil from the Persian Gulf has rendered these quotas largely theoretical for several members. Non-OPEC+ production from the Atlantic Basin (U.S., Brazil, and Guyana) is pushing toward record levels, but infrastructure bottlenecks at Gulf Coast terminals are limiting the speed at which this "new" oil can reach hard-hit Asian markets.

**Geopolitical Risks and Shipping Disruptions**
The primary driver of market panic is the **near-total shutdown of the Strait of Hormuz**. Naval engagements and missile strikes have made the waterway impassable for commercial tankers, effectively trapping the output of Saudi Arabia, Kuwait, Iraq, and the UAE. While some oil is being rerouted through Saudi Arabia’s East-West Pipeline to the Red Sea, these alternative routes are operating at maximum capacity and are themselves under threat of drone activity. The U.S. naval blockade of Iranian exports has further tightened the market, though reports suggest that some "dark fleet" volumes continue to move toward China under extreme risk.

**US SPR and Emergency Policy Signals**
The U.S. Strategic Petroleum Reserve (SPR) stands at approximately **412 million barrels**, having been slowly replenished to roughly 58% capacity under the current administration's "Energy Dominance" policy. However, the focus has shifted abruptly from refilling to emergency drawdown. There are intense rumors in Washington and Paris that the **IEA is preparing to announce a coordinated release of up to 400 million barrels**—the largest in history—to prevent a global economic collapse. As of today, March 8, the market is pricing in the high probability of this announcement, which would include a U.S. contribution of roughly 172 million barrels via an emergency exchange program.

**Factors for a Sudden Large Move (Next 5–10 Days)**
*   **IEA Announcement:** A formal confirmation of the 400mb coordinated release could cause a sudden **$15–$20 "relief drop"** in WTI as liquidity returns to the market.
*   **Infrastructure Sabotage:** Any confirmed damage to the East-West Pipeline or major Saudi processing facilities (e.g., Abqaiq) would likely push Brent/WTI toward **$180+** instantly.
*   **Ceasefire Rumors:** Any diplomatic signal regarding a "maritime corridor" or a pause in hostilities between the U.S./Israel and Iran would trigger a massive liquidation of long positions.
*   **Refinery Constraints:** With Middle Eastern crude cut off, complex refineries in Asia may begin "economic shutdowns" due to lack of feedstock, potentially causing a collapse in crude demand that paradoxically sends refined product prices (diesel/jet fuel) even higher while crude softens.

</details>



---
### Mar 16, 2026 — WTI $93.50 /bbl

| | |
|---|---|
| **Prediction** | P(up > +$5) = **42%**  `████░░░░░░` |
| **Confidence** | Medium  `███░░` |
| **Rationale** | The market is currently in a volatile tug-of-war between a historic 10-15 mb/d supply disruption and a massive 400-million-barrel global SPR release. While the SPR intervention has successfully triggered a $20 retracement from recent highs, the risk of military escalation or failed tanker escorts in the Strait of Hormuz remains a high-probability upside catalyst. Given the extreme daily volatility, a $5 move represents a return to the mean of the current conflict-driven price regime. |
| **Key signals** | U.S. Navy tanker escort outcomes · Iranian retaliation against regional infrastructure · SPR liquidity absorption rates |
| **Actual outcome** | 🟢 **No shock** — price moved **-5.37 /bbl** over 5 days |
| **Verdict** | ✅ **Correct** — predicted no shock, price moved only -5.37 |
| **Brier score** | 0.176 🟡 ok *(0 = perfect, 0.25 = random)* |

<details><summary>📰 Oil market context used (cutoff: 2026-03-15) — click to expand</summary>

### **Oil Market Intelligence Briefing: March 15, 2026**

**Price Action & Market Sentiment**
WTI crude is currently trading near **$96.00/bbl**, while Brent remains elevated above **$103.00/bbl**. The market is characterized by extreme volatility following the launch of a major U.S.-Israeli military operation against Iran on February 28, 2026. Prices initially spiked to a peak of **$113.13** in early March but have recently retraced following a massive emergency intervention by the International Energy Agency (IEA). Despite this slight cooling, the Brent-WTI spread has widened to roughly **$12/bbl**, reflecting the disproportionate impact of Middle Eastern supply disruptions on international benchmarks and surging shipping insurance costs.

**OPEC+ Policy & Supply Outlook**
OPEC+ has maintained its decision to **pause production increases** through the end of Q1 2026, citing seasonal demand weakness and the extreme uncertainty of the current conflict. However, the alliance’s internal cohesion is under severe strain; the **UAE has announced its intention to exit OPEC** effective May 1, 2026, a move that threatens to dismantle the group’s long-term spare capacity management. Currently, an estimated **10–15 million barrels per day (mb/d)** of production from the Persian Gulf is at risk or shut-in due to the effective closure of the Strait of Hormuz, leading to the largest supply shock in modern history.

**Geopolitical Flashpoints: The Persian Gulf War**
The **Strait of Hormuz remains a de facto no-go zone** for commercial tankers following the escalation of hostilities between the U.S.-led coalition and Iran. Reports of underwater mines and drone attacks on shipping have halted approximately 20% of global oil flows. While the U.S. Navy has begun operations to escort commercial vessels, the "tanker war" dynamics have effectively neutralized the export capacity of Iraq, Kuwait, and the UAE. Any sign of Iranian retaliation against Saudi energy infrastructure or a failed escort mission could send prices toward the **$140–$150** range within days.

**US Policy & SPR Intervention**
In a historic move on March 11, 2026, President Trump authorized the release of **172 million barrels** from the Strategic Petroleum Reserve (SPR) as part of a coordinated 400-million-barrel IEA action. To manage the fiscal impact, the Department of Energy is utilizing a "swap" strategy—selling immediate-delivery barrels at current highs ($96+) while simultaneously contracting to repurchase them for 2027 delivery at significantly lower futures prices (~$71). This temporal arbitrage is intended to flood the market with liquidity in the next 120 days, though critics warn that the SPR is now at its lowest level since its inception, leaving the U.S. with a dangerously thin margin for further shocks.

**Short-Term Outlook (5–10 Days)**
The next 5–10 days are critical for WTI. A **sudden large move** is likely depending on two factors:
1. **Escalation/Retaliation:** If Iran successfully strikes Atlantic Basin production or key transit hubs outside the Gulf, WTI will likely break past **$120/bbl**.
2. **Escort Success:** If the first wave of U.S.-escorted tankers successfully transits the Strait of Hormuz without incident, a "relief sell-off" could see WTI drop toward **$85/bbl** as the immediate scarcity premium evaporates. 
Traders should also watch for an emergency OPEC+ meeting, as the group faces immense pressure to fill the void left by the Persian Gulf blockade.

</details>



---
### Mar 23, 2026 — WTI $88.13 /bbl

| | |
|---|---|
| **Prediction** | P(up > +$5) = **40%**  `████░░░░░░` |
| **Confidence** | Medium  `███░░` |
| **Rationale** | The market is currently undergoing a sharp correction as it absorbs the impact of a historic 400-million-barrel coordinated SPR release, which has temporarily offset the panic from the Hormuz blockade. However, the underlying deficit of 20 million b/d remains fundamentally unresolved, and the recent $10 drop to $88.13 may be viewed as an overcorrection by some participants. A $5 bounce is highly plausible if the physical delivery of SPR barrels faces logistical hurdles or if military actions expand to inland infrastructure. |
| **Key signals** | SPR discharge and delivery logistics · Potential military strikes on Saudi Ghawar or Abqaiq infrastructure · Emerging demand destruction data from major Asian importers |
| **Actual outcome** | 🔴 **SHOCK** — price moved **+14.75 /bbl** over 5 days |
| **Verdict** | ❌ **Missed** — predicted no shock, but price rose +$14.75 |
| **Brier score** | 0.360 🔴 poor *(0 = perfect, 0.25 = random)* |

<details><summary>📰 Oil market context used (cutoff: 2026-03-22) — click to expand</summary>

### **Oil Market Intelligence Briefing**
**Date:** March 22, 2026
**Subject:** Global Supply Crisis and Strategic Intervention

#### **Price Action and Immediate Context**
WTI and Brent crude prices are currently trading at extreme premiums following the most severe supply shock in modern history. Since the initiation of U.S.-Israeli military action against Iran on February 28 and the subsequent retaliatory closure of the Strait of Hormuz on March 4, Brent has surged past the **$120/bbl** mark, with WTI trailing closely in the **$110–$115/bbl** range. Market volatility is at decadal highs as traders price in the total loss of approximately **20 million barrels per day (b/d)** of crude and refined products that typically transit the Strait. While prices have stabilized slightly from their initial peak due to intervention news, the "war-risk premium" remains the dominant driver of the current curve.

#### **Supply Disruptions and OPEC+ Status**
The physical market is in a state of emergency. Gulf producers, including Saudi Arabia, Iraq, Kuwait, and the UAE, have been forced to shut in an estimated **10 million b/d** of production as storage capacity in the region reaches its limits with no viable export outlets. QatarEnergy has declared *force majeure* on all LNG and condensate exports. OPEC+ as a cohesive entity is effectively paralyzed; while the group had previously agreed to extend production cuts through 2026, those quotas are now irrelevant compared to the massive involuntary shut-ins. Furthermore, the **UAE’s recent announcement to exit OPEC** (effective May 1) has introduced a secondary layer of long-term structural uncertainty regarding future spare capacity and cartel unity.

#### **Geopolitical Risks and Shipping**
The Strait of Hormuz remains functionally closed to commercial traffic. War-risk insurance premiums for the Persian Gulf have become prohibitive, and several major tanker fleets have suspended all regional operations. Beyond the blockade, there are credible reports of damage to regional refining infrastructure—totaling roughly **3 million b/d** of capacity—due to asymmetric attacks. Shipping lanes in the Red Sea are also seeing increased congestion as Saudi Arabia attempts to divert limited volumes through the East-West Pipeline to Yanbu, though this route can only mitigate a fraction of the lost Hormuz volumes.

#### **Strategic Reserve Intervention**
The primary bearish counterweight is the massive, coordinated emergency release announced by the IEA on March 11. Member nations have committed to releasing **400 million barrels** from strategic stocks, the largest such action in history. The U.S. portion of this release, totaling **172 million barrels**, is scheduled to begin hitting the market this week. While this provides a temporary physical buffer, analysts warn that the U.S. Strategic Petroleum Reserve (SPR) is being drawn down from an already depleted level of approximately **415 million barrels**, potentially leaving the global market vulnerable if the conflict extends beyond a 90-day window.

#### **Short-Term Volatility Catalysts (5–10 Day Outlook)**
WTI is susceptible to a **$15–$20 move** in either direction over the next 10 days based on the following triggers:
*   **Escalation/De-escalation Signals:** Any indication of a ceasefire or, conversely, a widening of the conflict to include strikes on Saudi "Ghawar" or "Abqaiq" infrastructure would cause immediate price gaps.
*   **SPR Delivery Logistics:** The market is hyper-focused on the actual "discharge rate" of the 172M barrel U.S. release. Any logistical delays in moving these barrels to Gulf Coast refiners will cause a sharp upward correction in WTI.
*   **Demand Destruction Data:** Early indicators of "panic-driven" demand destruction in Asia and Europe are emerging. If March 2026 consumption data shows a contraction greater than the current 1M b/d estimate, prices may see a sharp "relief" sell-off.

</details>


In [12]:
# ── Act 6 summary chart ────────────────────────────────────────────────────────
#
# Top panel  — dot+line chart: P(upward shock) for Prophet & Analyst Agent
#              Shock-outcome weeks highlighted with a red column background
#              so the audience can instantly read "model said X, this happened"
#
# Bottom panel — cumulative mean Brier score (lower = better)
#
# Sized narrow (640 px wide) for pptx embedding.

METHOD_COLORS = {"Prophet": CLR_PROPHET, "Analyst Agent": CLR_LLMP_CTX}
METHOD_SYMBOL = {"Prophet": "square",    "Analyst Agent": "circle"}
METHOD_DASH   = {"Prophet": "dot",       "Analyst Agent": "solid"}

origins_ordered = [o.strftime("%Y-%m-%d") for o in SHOCK_ORIGINS]
outcome_by_key  = {
    key: int(binary_df[(binary_df["origin"] == key) & (binary_df["method"] == "Prophet")].iloc[0]["outcome"])
    for key in origins_ordered
}
shock_indices = [i for i, k in enumerate(origins_ordered) if outcome_by_key[k] == 1]

fig = psp.make_subplots(
    rows=2, cols=1,
    row_heights=[0.58, 0.42],
    vertical_spacing=0.20,
    subplot_titles=[
        "P(WTI up > +$5 /bbl in 5 trading days)",
        "Cumulative mean Brier score (lower = better)",
    ],
)

# ── Shock-outcome column shading (both panels) ────────────────────────────────
# Categorical x-axes map category n → integer index n, so use numeric offsets.
for i in shock_indices:
    for row_n, (y0, y1) in [(1, (-0.12, 1.08)), (2, (0.0, 0.30))]:
        fig.add_shape(
            type="rect", layer="below",
            xref=f"x{'' if row_n == 1 else row_n}",
            yref=f"y{'' if row_n == 1 else row_n}",
            x0=i - 0.48, x1=i + 0.48,
            y0=y0, y1=y1,
            fillcolor="rgba(214,39,40,0.12)",
            line_width=0,
        )
    # Label at the top of the shaded column (use string category value)
    fig.add_annotation(
        x=origins_ordered[i], y=1.06, text="<b>SHOCK</b>",
        showarrow=False,
        font=dict(size=9, color=CLR_CONFLICT),
        xref="x", yref="y",
    )

# ── Row 1: probability dot + line ─────────────────────────────────────────────
for method in ["Analyst Agent", "Prophet"]:
    sub = binary_df[binary_df["method"] == method].sort_values("origin_dt")
    fig.add_trace(go.Scatter(
        x=sub["origin"],
        y=sub["prob"],
        name=method,
        mode="lines+markers",
        line=dict(color=METHOD_COLORS[method], width=2.5, dash=METHOD_DASH[method]),
        marker=dict(size=10, symbol=METHOD_SYMBOL[method]),
        legendgroup=method,
        showlegend=True,
        hovertemplate="%{x}<br>P(up)=%{y:.0%}<extra>" + method + "</extra>",
    ), row=1, col=1)
    # Probability label above each dot
    yshift = 12 if method == "Analyst Agent" else -14
    for _, r in sub.iterrows():
        fig.add_annotation(
            x=r["origin"], y=r["prob"],
            text=f"{r['prob']:.0%}",
            showarrow=False,
            font=dict(size=8, color=METHOD_COLORS[method]),
            yshift=yshift,
            row=1, col=1,
        )

fig.add_hline(
    y=0.5, line=dict(color="#d0d0d0", dash="dot", width=1.2),
    row=1, col=1,
)

fig.update_yaxes(
    title_text="P(up > +$5)", range=[-0.08, 1.14],
    tickformat=".0%", dtick=0.25, row=1, col=1,
)
fig.update_xaxes(tickangle=-30, row=1, col=1)

# ── Row 2: cumulative mean Brier ───────────────────────────────────────────────
for method in ["Analyst Agent", "Prophet"]:
    sub       = binary_df[binary_df["method"] == method].sort_values("origin_dt")
    cum_brier = sub["brier"].expanding().mean().values
    fig.add_trace(go.Scatter(
        x=sub["origin"].values,
        y=cum_brier,
        name=method,
        mode="lines+markers",
        line=dict(color=METHOD_COLORS[method], width=2.5, dash=METHOD_DASH[method]),
        marker=dict(size=8, symbol=METHOD_SYMBOL[method]),
        legendgroup=method,
        showlegend=False,
        hovertemplate="%{x}<br>Cumul. Brier: %{y:.3f}<extra>" + method + "</extra>",
    ), row=2, col=1)

fig.add_hline(
    y=0.25, line=dict(color="#aaa", dash="dot", width=1.5),
    annotation_text="0.25 random ceiling",
    annotation_position="top right",
    annotation_font=dict(size=9, color="#888"),
    row=2, col=1,
)
fig.update_yaxes(
    title_text="Brier score", range=[0, 0.30],
    row=2, col=1,
)
fig.update_xaxes(tickangle=-30, row=2, col=1)

fig.update_layout(
    title=dict(
        text="Analyst Agent vs. Prophet — Upward Shock (Feb–Mar 2026)",
        x=0.5, font=dict(size=13),
    ),
    height=520, width=640,
    template="plotly_white",
    # Force categorical so the 8 weekly dates each get their own column
    xaxis=dict(type="category", tickangle=-35, showgrid=False),
    xaxis2=dict(type="category", tickangle=-35, showgrid=False),
    legend=dict(
        orientation="h", yanchor="bottom", y=1.04, xanchor="right", x=1,
        font=dict(size=11),
    ),
    margin=dict(t=80, b=65, l=60, r=35),
)
fig.update_yaxes(showgrid=True, gridcolor="#ececec", gridwidth=0.7)
fig.show()